# Lesson 4, Part 4

In [ ]:
%pip install ipycc
from ipycc.turtle import Turtle, showscreen, clearscreen
import numpy as np, matplotlib.pyplot as plt

We will use Python to approximately verify the shell theorem discussed in Level 4 Part 2. 
Let's start by considering a ring of radius *R*, an object *r* > *R* away from its center, and an arbitrary point 
on the ring with coordinate *x* in the direction of the object. Try different values, including negative *x*.

In [ ]:
# pulled by mass on ring with coordinate x in direction of object
def diagram_ring_pull(R,r,x):
    t = Turtle()
    t.speed('fastest')
    t.home(); t.dot()  # places dot at home
    t.write('0')                            # draw straight from center to (r,0)
    t.forward(r); t.dot(); t.write('  (r,0)')
    t.backward(r-R); t.write('  R')
    t.penup(); t.goto(R,0); t.pendown()
    t.color('blue'); t.left(90); t.circle(R)    # draw circle of radius R
    t.penup(); t.home(); t.pendown()
    t.color('red'); y = (R**2-x**2)**0.5    # identify point (x,y) on circle
    t.goto(x/2,y/2); t.write('  R')
    t.goto(x,y); t.dot(); t.write('(x,y)  ')    # draw triangle and place markers
    t.goto(x,y/2); t.write('  y')
    t.goto(x,0); t.write('  x')
    t.home();
    t.penup(); t.goto(r,0); t.pendown()
    t.color('purple')                       # draw line from (r,0) to (x,y)
    t.goto((x+r)/2,y/2); t.write('  D')
    t.goto(x,y)
    t.hideturtle()

def diagram_ring_pull(R,r,x):
    shift = -80
    t = Turtle()
    t.speed('fastest')
    t.hideturtle()
    t.penup()
    t.goto(shift,0)
    t.pendown()
    t.dot()  # places dot at mass center
    t.write('0')                              # draw straight from center to (r,0)
    t.forward(r); t.dot(); t.write('  (r,0)')
    t.backward(r-R); t.write('  R')
    t.penup(); t.goto(shift+R,0); t.pendown()
    t.color('blue'); t.left(90); t.circle(R)  # draw circle of radius R
    t.penup(); t.goto(shift,0); t.pendown()
    t.color('red'); y = (R**2 - x**2)**0.5       # identify point (x,y) on circle
    t.goto(x/2+shift, y/2); t.write('  R')
    t.goto(shift+x, y); t.dot(); t.write('(x,y)  ')    # draw triangle and place markers
    t.goto(shift+x, y/2); t.write('  y')
    t.goto(shift+x, 0); t.write('  x')
    t.goto(shift, 0);
    t.penup(); t.goto(shift+r,0); t.pendown()
    t.color('purple')                       # draw line from (r,0) to (x,y)
    t.goto(shift + (x + r)/2,y/2); t.write('  D')
    t.goto(shift+x,y)
    
showscreen()
clearscreen()
R,r,x = 120,240,80
diagram_ring_pull(R,r,x)

Assuming that mass is equally distributed around the ring, upward pulls will be cancelled out by downward pulls. 
However, the horizontal components will not cancel out. Let's compute the pull toward the center for various points on the ring. 
It will be highest for points nearest to the object.

In [ ]:
def pull_to_center(R, r, num=500, GM=1000):
    x = np.linspace(-R,R,num)
    y = np.sqrt(R**2 - x**2)            # compute y-coordinate on ring given x_coordinate
    D = np.sqrt((r - x)**2 + y**2)      # compute distance D between (x,y) and object
    # D = np.sqrt(r**2 - 2*r*x + R**2)  # computes D in one step, without needing y
    pull_to_point = GM / D**2           # Newton's law of gravity, directed toward (x,y)
    pull_x = (r-x)/D * pull_to_point    # x-component of the pull 
    # pull_x = GM * (r - x) / D**3      # summarizes results of previous 2 steps
    return x,pull_x 

x,pull_x = pull_to_center(R,r)
plt.plot(x,pull_x)
plt.title('Gravitational pull toward center')
plt.show()

Next, imagine that we rotate the ring to make a hollow shell. 
The mass with *x*-coordinates closest to *x*[i] forms a slanted band with average radius *y*[i].
In Level 2 Part 4 we learned Archimedes' proof that the surface areas of equally spaced bands don't vary with *x*[i].
Hence their masses are equal too. The code below verifies that, with tiny errors at the edge due to our approximations.

In [ ]:
def relative_mass(R,x):
    y = np.sqrt(R**2 - x**2)
    gap_x = x[2:] - x[:-2]   # gap around x
    gap_y = y[2:] - y[:-2]   # gap around y
    distance_between = np.sqrt(gap_x**2 + gap_y**2)
    weight = np.zeros(len(x))
    weight[1:-1] = distance_between * y[1:-1]
    average_weight = np.sum(weight) / len(weight)
    return weight / average_weight

R,r = 100,200
x,pull_x = pull_to_center(R,r)
rel_mass = relative_mass(R,x)
plt.plot(x,rel_mass)
plt.xlabel('Distance from center')
plt.ylabel('Relative mass of shell rings')
plt.show()

Let's compute the net gravity for the shell using either the simple average or the weighted average. 
Each is very close to what the gravity would be if all mass were concentrated at the center. 
Since we can think of a sphere as a nesting of thin hollow shells, 
this applies for the sphere as a whole.

In [ ]:
def rel_deviation(estimate,r,GM):
    return estimate / (GM / r**2) - 1

def calculate_gravity(R,r,num=500,GM=1):
    x,pull_x = pull_to_center(R,r,num,GM)
    # simple_avg = np.sum(pull_x) / len(x)
    simple_avg = np.mean(pull_x)   # numpy.mean() works exactly like preceding
    rel_dev_simple = rel_deviation(simple_avg,r,GM)
    rel_mass = relative_mass(R,x)
    wtd_avg = np.sum(pull_x * rel_mass)
    rel_dev_wtd = rel_deviation(wtd_avg,r,GM)
    return simple_avg,wtd_avg,rel_dev_simple,rel_dev_wtd

R,r = 100,200
simple_avg,wtd_avg,rel_dev_simple,rel_dev_wtd = calculate_gravity(R,r,num=500,GM=10)
print('Fractional deviation from gravity if all mass at center')
print('  simple average: ',round(rel_dev_simple,5))
print('weighted average:',round(rel_dev_wtd,5))

Imagine that the object is inside the shell, so that 0 ≤ *r* < *R*. A diagram shows that the same kind of computation applies.

In [ ]:
showscreen()
clearscreen()
R,r,x = 200,125,-100
diagram_ring_pull(R,r,x)

For objects inside a uniform spherical shell, the net gravity is zero.
Our calculations approximately verify it.
The result helps explain why trillions of stars outside our solar system 
have little net gravitational impact on us.

In [ ]:
R,r,x = 200,125,-100
simple_avg,wtd_avg,rel_dev_simple,rel_dev_wtd = calculate_gravity(R,r,num=500,GM=10)
print('Fractional deviation from gravity if all mass at center')
print('  simple average:',round(rel_dev_simple,5))
print('weighted average:',round(rel_dev_wtd,5))

Let's now develop code for computing osculating circles, 
which match a function *y*=*f*(*x*) in value, 
slope and curvature for a given *x*.
Let's start by computing an approximate first derivative 
for sequences of *x* and *y*, 
and then compute an approximate second derivative through iteration. 
A function that calls itself is known as **recursive**.

In [ ]:
# estimate first derivative or (if isTwo) first and second derivatives
def approx_derivatives(x,y,isTwo=False):
    deriv = np.zeros(len(x))
    deriv[1:-1] = (y[2:] - y[:-2]) / (x[2:] - x[:-2])
    deriv[0] = (y[1]-y[0]) / (x[1] - x[0])
    deriv[-1] = (y[-1]-y[-2]) / (x[-1] - x[-2])
    if isTwo:  # then compute second derivative too and return both
        return deriv, approx_derivatives(x,deriv,False) 
    else:      # otherwise return only first derivative
        return deriv 

x = np.linspace(-5,5,500)
y = x**2
deriv1,deriv2 = approx_derivatives(x,y,True)
plt.title('y=x**2 and its first and second derivatives')
plt.plot(x,y,x,deriv1,x,deriv2)
plt.legend(['y',"y'",'y"'])  # need to alternate " and ' to avoid error
plt.show()

Let's use this info to compute the radii and centers of the osculating circles. 
The most noticeable problem is that the radii tend to dwarf the function wherever it gets nearly linear. 
The reciprocal of the radius, known as the curvature, generally maintains a much tighter range.

In [ ]:
def osculating_circle(x,y,deriv1,deriv2):
    factor = np.sqrt(1 + deriv1**2)
    radius = factor**3 / deriv2    # derived in text, and allows for radius < 0
    # slope of radial line from tangent = -1/deriv1
    center_x = x - deriv1 * radius / factor
    center_y = y + radius / factor
    return radius,center_x,center_y

x = np.linspace(-5,5,500)
y = x**2
deriv1,deriv2 = approx_derivatives(x,y,True)
radius,center_x,center_y = osculating_circle(x,y,deriv1,deriv2)
plt.figure(constrained_layout=True)  # this command keeps titles and axes from overlapping
plt.subplot(3,1,1)
plt.title('First Derivative of y')
plt.plot(x,deriv1)
plt.subplot(3,1,2)
plt.title('Radius of Osculating Circles')
plt.plot(x,radius)
plt.subplot(3,1,3)
plt.title('Curvature = 1/Radius')
plt.plot(x,1/radius)
plt.show()

Here is another set of osculating circles.

In [ ]:
x = np.linspace(0,4,250)
y = x * (x-1) * (x-2) * (x-3) * (x-4)
plt.figure(constrained_layout=True) 
plt.subplot(3,1,1)
plt.title('Function')
plt.plot(x,y)
deriv1,deriv2 = approx_derivatives(x,y,True)
radius,center_x,center_y = osculating_circle(x,y,deriv1,deriv2)
plt.subplot(3,1,2)
plt.title('Radius of Osculating Circles')
plt.plot(x,radius)
plt.subplot(3,1,3)
plt.title('Curvature = 1/Radius')
plt.plot(x,1/radius)
plt.show()

Numpy makes it easy to chart the osculating circles. However, the variations in centers and scale overwhelm the function itself. 

In [ ]:
def rim_circle(radius,center_x,center_y):
    rim_x = np.linspace(-radius,radius,100)
    rim_y = np.sqrt(radius**2 - rim_x**2)           # top half of circle around origin
    rim_x = np.concatenate((rim_x,rim_x[-2::-1]))  
    rim_y = np.concatenate((rim_y,-rim_y[-2::-1]))  # adds bottom half of circle 
    rim_x += center_x   # shifts center
    rim_y += center_y   # shifts center
    return rim_x,rim_y

x = np.linspace(0,4,250)
y = x * (x - 1) * (x - 2) * (x - 3) * (x - 4)
deriv1,deriv2 = approx_derivatives(x,y,True)
radius,center_x,center_y = osculating_circle(x,y,deriv1,deriv2)
rim_x,rim_y = rim_circle(radius,center_x,center_y)
fig = plt.figure()
ax = fig.add_axes([0,0,1,1])
ax.set_aspect('equal', adjustable='box')    # without this circles will look like ellipses
ax.plot(rim_x[:,::10],rim_y[:,::10])
bound = 15 
ax.set_ylim(-bound,bound)
ax.set_xlim(-bound,bound)
plt.show()

The code here plots a few osculating circles with smaller radii.

In [ ]:
x = np.linspace(0,4,250)
y = x * (x - 1) * (x - 2) * (x - 3) * (x - 4)
deriv1,deriv2 = approx_derivatives(x,y,True)
radius,center_x,center_y = osculating_circle(x,y,deriv1,deriv2)
plt.title('Function and Some Osculating Circles')
plt.axis('equal')
plt.plot(x,y)
range_y = max(y) - min(y)
# range_y = np.ptp(y)   # numpy function ptp()  "peak to peak" does the same
for k in range(1,10):
    ix = len(x) * k // 10
    if np.abs(radius[ix]) < 2 * range_y:
        rim_x,rim_y = rim_circle(radius[ix],center_x[ix],center_y[ix])
        plt.plot(rim_x,rim_y)
plt.show()

Let's build some splines. 
We'll start with a linear spline, 
which connects neighboring points with straight lines and creates a jagged plot.

In [ ]:
def linear_spline(x,y):
    for i in range(len(x)-1):
        plt.plot((x[i],x[i+1]),(y[i],y[i+1]))
    plt.show()
        
x = np.linspace(0,4,20)   # only 20 points used here, versus 250 before
y = x * (x-1) * (x-2) * (x-3) * (x-4)
plt.title('Linear Spline')
linear_spline(x,y)

To make plots less jagged, let's try to match slopes. 
Thes code below fits quadratic segments between neighboring points, with slope at the first point matching the approximate first derivative there. 
It is less jagged than the linear spline but still not smooth.

In [ ]:
def quadratic_spline(x,y):
    slope = approx_derivatives(x,y)
    for i in range(len(x)-1):
        t = np.linspace(0,1,20)
        xx = t * (x[i+1] - x[i])  + x[i] 
        C = slope[i] * (x[i+1] - x[i]) + y[i] - y[i+1]
        yy = y[i] + (y[i+1] - y[i]) * t + C * t * (1 - t) 
        plt.plot(xx,yy)
    plt.show()

x = np.linspace(0,4,20)
y = x * (x - 1) * (x - 2) * (x - 3) * (x - 4)
plt.title('Quadratic Spline')
quadratic_spline(x,y)

A cubic spline between two points can match slopes at both points. 
Now the fit looks smooth.

In [ ]:
def cubic_spline(x,y):
    slope = approx_derivatives(x,y)
    for i in range(len(x)-1):
        t = np.linspace(0,1,20)
        xx = t * (x[i+1] - x[i])  + x[i] 
        C = slope[i] * (x[i+1] - x[i]) + y[i] - y[i+1]
        D = -slope[i+1] * (x[i+1] - x[i]) + y[i+1] - y[i]
        yy = y[i] + (y[i+1] - y[i]) * t + (C * (1-t) + D *t) * t * (1-t) 
        plt.plot(xx,yy)
    plt.show()

x = np.linspace(0,4,20) 
y = x * (x-1) * (x-2) * (x-3) * (x-4)
plt.title('Cubic Spline')
cubic_spline(x,y)